# Практична робота №2. Якість, валідація та очищення IoT-даних

У цій роботі потрібно підготувати сирі IoT-події до подальшої аналітики.
Вхідний `raw_iot_events.jsonl` є синтаксично коректним JSONL-файлом, але окремі події можуть містити дефекти якості.

Результатом роботи мають бути файли:

```text
results/practical_02/clean_iot_events.parquet
results/practical_02/data_quality_issues.csv
results/practical_02/rejected_iot_events.jsonl
results/practical_02/cleaning_summary.json
```

## 1. Підготовка середовища та шляхів

Ноутбук має працювати і в JupyterLab усередині контейнера, і при локальному запуску з кореня репозиторію.

In [ ]:
from pathlib import Path
import json
from collections import Counter

import polars as pl

In [ ]:
ROOT = Path("/workspace") if Path("/workspace").exists() else Path.cwd()
if not (ROOT / "data" / "input").exists() and (ROOT.parent / "data" / "input").exists():
    ROOT = ROOT.parent

INPUT_DIR = ROOT / "data" / "input"
RESULTS_DIR = ROOT / "results" / "practical_02"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_EVENTS_PATH = INPUT_DIR / "raw_iot_events.jsonl"
METADATA_PATH = INPUT_DIR / "metadata.json"
DEVICE_REGISTRY_PATH = INPUT_DIR / "device_registry.csv"
METRIC_CATALOG_PATH = INPUT_DIR / "metric_catalog.csv"
DEVICE_TYPE_METRICS_PATH = INPUT_DIR / "device_type_metrics.csv"

CLEAN_PATH = RESULTS_DIR / "clean_iot_events.parquet"
ISSUES_PATH = RESULTS_DIR / "data_quality_issues.csv"
REJECTED_PATH = RESULTS_DIR / "rejected_iot_events.jsonl"
SUMMARY_PATH = RESULTS_DIR / "cleaning_summary.json"

ROOT

## 2. Завантаження вхідних файлів

У цій моделі `raw_iot_events.jsonl` можна читати стандартним засобом Polars, оскільки кожен рядок є валідним JSON-об'єктом.  
Відсутні поля у частині подій будуть прочитані як `null`.

In [ ]:
with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

raw_events = pl.read_ndjson(RAW_EVENTS_PATH).with_row_index("source_row", offset=1)
device_registry = pl.read_csv(DEVICE_REGISTRY_PATH)
metric_catalog = pl.read_csv(METRIC_CATALOG_PATH)
device_type_metrics = pl.read_csv(DEVICE_TYPE_METRICS_PATH)

print("Raw events:", raw_events.height)
print("Devices:", device_registry.height)
print("Metrics:", metric_catalog.height)
raw_events.head()

## 3. Очікувана схема сирої події

Базові поля події:

```text
event_id
event_ts
device_id
event_type
metric
value
```

Усі записи без одного з цих полів, крім службового `source_row`, мають бути відхилені.

In [ ]:
REQUIRED_EVENT_FIELDS = [
    "event_id",
    "event_ts",
    "device_id",
    "event_type",
    "metric",
    "value",
]

ALLOWED_EVENT_TYPES = {"telemetry", "status", "network"}

ISSUE_TYPES = [
    "duplicate_event_id",
    "missing_required_field",
    "invalid_timestamp",
    "unknown_device_id",
    "invalid_metric_for_device_type",
    "out_of_range_value",
]

## 4. Службові колекції для результатів

Рекомендовано накопичувати проблеми та відхилені записи у списках словників, а в кінці перетворити їх у файли.

In [ ]:
issues = []
rejected_records = []

def add_issue(event_id, issue_type, issue_detail, action):
    issues.append({
        "event_id": str(event_id),
        "issue_type": issue_type,
        "issue_detail": issue_detail,
        "action": action,
    })

def add_rejected(raw_record, issue_type, issue_detail, action):
    event_id = raw_record.get("event_id", "")
    rejected_records.append({
        "event_id": str(event_id),
        "issue_type": issue_type,
        "issue_detail": issue_detail,
        "action": action,
        "raw_record": raw_record,
    })

## 5. Обробка дублікатів `event_id`

Правило: перша поява події залишається, додаткові повні копії вилучаються як `removed_duplicate`.

У поточному датасеті генератор створює саме повні копії рядків.

In [ ]:
# TODO:
# 1. Знайдіть повторні event_id.
# 2. Переконайтеся, що додаткові появи є повними копіями.
# 3. Залиште першу появу.
# 4. Додайте рядки до issues та rejected_records для вилучених копій.

events_step = raw_events

# Приклад очікуваної ідеї:
# duplicate_mask = ...
# events_step = ...

## 6. Перевірка обов'язкових полів

Якщо бракує хоча б одного обов'язкового поля, запис відхиляється.  
Відновлювати `event_ts`, `device_id`, `event_type`, `metric` або `value` не потрібно.

In [ ]:
# TODO:
# 1. Для кожного required field знайдіть null.
# 2. Такі записи додайте до issues та rejected_records.
# 3. Видаліть їх із подальшої обробки.

# Рекомендований формат issue_detail:
# missing_event_ts
# missing_device_id
# missing_event_type
# missing_metric
# missing_value

## 7. Нормалізація та перевірка `event_ts`

Дозволені варіанти:

```text
2026-03-02T14:25:30Z   → коректний формат
2026/03/02 14:25:30    → виправний формат
2026-02-30T14:25:30Z   → неможлива дата, запис відхиляється
```

Після очищення `event_ts` у Parquet має бути типу `Datetime`.

In [ ]:
from datetime import datetime, timezone

def parse_event_ts(value):
    """Повертає tuple: (parsed_datetime або None, issue_detail або None, action або None)."""
    if value is None:
        return None, "missing_event_ts", "rejected"

    text = str(value)

    try:
        dt = datetime.fromisoformat(text.replace("Z", "+00:00")).astimezone(timezone.utc)
        return dt, None, None
    except ValueError:
        pass

    try:
        dt = datetime.strptime(text, "%Y/%m/%d %H:%M:%S").replace(tzinfo=timezone.utc)
        return dt, "invalid_format", "repaired"
    except ValueError:
        return None, "impossible_date", "rejected"

In [ ]:
# TODO:
# 1. Застосуйте parse_event_ts до event_ts.
# 2. invalid_format нормалізуйте до Datetime і позначте action=repaired.
# 3. impossible_date відхиліть.
# 4. Перевірте, що час входить у межі metadata["period_start"]..metadata["period_end"].

## 8. Перевірка `device_id`

Правило: `device_id` має існувати в `device_registry.csv`.  
Невідомі пристрої не додаються до реєстру автоматично.

In [ ]:
# TODO:
# 1. Виконайте перевірку device_id через device_registry.
# 2. Записи з невідомими пристроями відхиліть.
# 3. issue_type = unknown_device_id
# 4. issue_detail можна сформувати як device_id=<значення>

## 9. Перевірка `event_type` та `metric`

Загальні правила:

```text
event_type ∈ telemetry/status/network
metric існує в metric_catalog.csv
```

Для telemetry-подій додатково:

```text
device_id → device_type → device_type_metrics.csv → допустимі metric
```

In [ ]:
# TODO:
# 1. Перевірте allowed event_type.
# 2. Перевірте наявність metric у metric_catalog.csv.
# 3. Для telemetry-подій перевірте відповідність metric типу пристрою.
# 4. issue_type = invalid_metric_for_device_type для неузгоджених telemetry-метрик.

## 10. Перевірка `value`

Для кожної метрики потрібно використати `metric_catalog.csv`:

```text
min_value <= value <= max_value
```

Для метрик з `value_type = int` значення має бути фактично цілим.

In [ ]:
# TODO:
# 1. Приєднайте min_value, max_value, value_type з metric_catalog.csv.
# 2. Відхиліть записи, де value поза межами.
# 3. Для int-метрик перевірте, що value не має дробової частини.
# 4. issue_type = out_of_range_value

## 11. Формування `clean_iot_events.parquet`

У фінальному Parquet мають бути тільки такі колонки:

```text
event_id
event_ts
device_id
event_type
metric
value
```

Службові колонки, наприклад `source_row`, не записуються.

In [ ]:
# TODO:
# clean_events = ...

# clean_events = clean_events.select([
#     "event_id",
#     "event_ts",
#     "device_id",
#     "event_type",
#     "metric",
#     "value",
# ])

# clean_events.write_parquet(CLEAN_PATH)

## 12. Формування `data_quality_issues.csv`

In [ ]:
# TODO:
# issues_df = pl.DataFrame(issues)
# issues_df.write_csv(ISSUES_PATH)

## 13. Формування `rejected_iot_events.jsonl`

Сюди потрапляють тільки записи, які не увійшли до очищеного набору:

```text
rejected
removed_duplicate
```

Виправлені записи сюди не додаються.

In [ ]:
# TODO:
# with REJECTED_PATH.open("w", encoding="utf-8") as f:
#     for row in rejected_records:
#         f.write(json.dumps(row, ensure_ascii=False) + "\n")

## 14. Формування `cleaning_summary.json`

Контрольні рівності:

```text
input_records = clean_records + rejected_records + removed_duplicate_records
clean_records = unchanged_records + repaired_records
```

In [ ]:
# TODO:
# action_counts = Counter(...)
# issue_counts = Counter(...)
# detail_counts = Counter(...)
#
# summary = {
#     "input_records": raw_events.height,
#     "clean_records": clean_events.height,
#     "unchanged_records": ...,
#     "repaired_records": ...,
#     "rejected_records": ...,
#     "removed_duplicate_records": ...,
#     "issues_by_type": dict(issue_counts),
#     "issues_by_detail": dict(detail_counts),
#     "actions_by_type": dict(action_counts),
# }
#
# with SUMMARY_PATH.open("w", encoding="utf-8") as f:
#     json.dump(summary, f, ensure_ascii=False, indent=2)

## 15. Самоперевірка

Після створення результатів запустіть у терміналі з кореня репозиторію:

```bash
python scripts/check_practical_02_outputs.py
```

Скрипт перевіряє структуру результатів і базові правила якості, але не замінює повної перевірки викладачем.

In [ ]:
# Необов'язково: запуск self-check прямо з ноутбука
# !python scripts/check_practical_02_outputs.py